# BDAP LEA: il peso della prevenzione nelle ASL italiane (2024)

Prima lettura analitica per una discussion `Analisi`.

- **Obiettivo**: calcolare l'incidenza percentuale della Prevenzione Collettiva e Sanita' Pubblica sulla spesa corrente di ogni ASL nel consuntivo 2024.
- **Dati**: mart `mart_spesa_enti_2024` (BDAP LEA).
- **Orizzonte**: consuntivo 2024, livello ente ASL.
- **Riferimento**: soglia LEA del 5% come minimo indicato.
- **Perimetro**: filtro su ASL/USL/ASP (escluse AO, AOU, enti regionali/atipici).
- **Nota**: il consuntivo 2024 e' l'ultimo anno completo disponibile.

In [1]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

pd.options.display.float_format = '{:,.1f}'.format

def find_workspace_root(start: Path) -> Path:
    # Sale dalla posizione corrente finche' non trova dataset-incubator
    candidate = start.resolve()
    if candidate.is_file():
        candidate = candidate.parent
    for _ in range(15):
        if (candidate / 'dataset-incubator').exists():
            return candidate
        parent = candidate.parent
        if parent == candidate:
            break
        candidate = parent
    raise RuntimeError(f'Workspace root non trovato (partito da {start})')

WORKSPACE_ROOT = find_workspace_root(Path.cwd())
mart_path = WORKSPACE_ROOT / 'dataset-incubator' / 'out' / 'data' / 'mart' / 'bdap_lea' / '2024' / 'mart_spesa_enti_2024.parquet'

if not mart_path.exists():
    raise FileNotFoundError(f'Mart non trovato: {mart_path.resolve()}')

con = duckdb.connect()


### 1. Incidenza della prevenzione per ASL

Per ogni ASL, confrontiamo il totale della Prevenzione Collettiva e Sanita' Pubblica con la spesa corrente totale (TOTALE GENERALE).

Il filtro include solo enti ASL/USL/ASP, escludendo Aziende Ospedaliere, Ospedaliere-Universitarie e enti regionali/atipici che per struttura contabile non sono confrontabili con il benchmark LEA del 5%.

In [2]:
query = """
WITH prevenzione AS (
    SELECT 
        descrizione_ente,
        round(sum(importo_totale), 0) as prev_totale
    FROM read_parquet(?)
    WHERE descrizione_voce_contabile = 'TOTALE PREVENZIONE COLLETTIVA E SANITA'' PUBBLICA'
    GROUP BY descrizione_ente
),
totale AS (
    SELECT 
        descrizione_ente,
        round(sum(importo_totale), 0) as spesa_corrente_totale
    FROM read_parquet(?)
    WHERE descrizione_voce_contabile = 'TOTALE GENERALE'
      AND (
          descrizione_ente LIKE '%SANITARIA LOCALE%'
          OR descrizione_ente LIKE '%UNITA'' SANITARIA LOCALE%'
          OR descrizione_ente LIKE '%SANITARIA PROVINCIALE%'
          OR descrizione_ente LIKE '%SOCIOSANITARIA LOCALE%'
      )
    GROUP BY descrizione_ente
),
calcolo AS (
    SELECT 
        t.descrizione_ente,
        t.spesa_corrente_totale,
        coalesce(p.prev_totale, 0) as prev_totale,
        round(coalesce(p.prev_totale, 0) / t.spesa_corrente_totale * 100, 2) as pct_prev
    FROM totale t
    LEFT JOIN prevenzione p ON t.descrizione_ente = p.descrizione_ente
)
SELECT * FROM calcolo
ORDER BY pct_prev DESC
"""

df = con.execute(query, [str(mart_path), str(mart_path)]).df()


In [3]:
df[['descrizione_ente', 'spesa_corrente_totale', 'prev_totale', 'pct_prev']].head(10)


### 2. Sintesi nazionale

Quante ASL superano la soglia del 5%?

In [4]:
summary = con.execute("""
SELECT 
    count(*) as totale_asl,
    count(*) FILTER (WHERE pct_prev >= 5) as asl_sopra_5,
    count(*) FILTER (WHERE pct_prev < 5) as asl_sotto_5,
    round(avg(pct_prev), 2) as pct_media,
    min(pct_prev) as pct_min,
    max(pct_prev) as pct_max,
    percentile_cont(0.5) WITHIN GROUP (ORDER BY pct_prev) as pct_mediana
FROM df
""").fetchdf()

summary


### 3. Prime letture

- **Solo 24 ASL su 70 (34.3%)** superano la soglia del 5%.
- La **mediana e' al 4.04%**, sotto il riferimento LEA.
- La **media e' 4.64%**.
- **46 ASL (65.7%)** stanno sotto il 5%.
- Il massimo tra le ASL e' 10.6% (ASL Napoli 1 Centro).

**Nota sul perimetro**: il filtro include solo enti ASL/USL/ASP (70 enti). Sono escluse le Aziende Ospedaliere/Ospedaliere-Universitarie (31 enti) e gli enti regionali/atipici (5 enti) che non hanno competenza di prevenzione collettiva o hanno struttura contabile diversa. Senza questo filtro il dataset conterrebbe 183 enti, di cui molti a 0% strutturale, falsando il confronto con la soglia LEA.

**Conclusione**: la prevenzione collettiva e' marginalizzata nella spesa corrente della maggioranza delle ASL italiane. Il fenomeno non e' limitato al Sud: diverse ASL del Nord sono sotto il 5%.

### 4. Grafico - distribuzione percentuale

Istogramma della percentuale di prevenzione per ASL, con la soglia LEA al 5% evidenziata.

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['pct_prev'], bins=15, color='#2563eb', edgecolor='white', linewidth=0.5)
ax.axvline(x=5, color='#c2410c', linestyle='--', linewidth=1.5, label='soglia LEA 5%')
ax.axvline(x=df['pct_prev'].median(), color='#059669', linestyle='--', linewidth=1.5, label=f"mediana {df['pct_prev'].median():.2f}%")

ax.set_title('Distribuzione % Prevenzione Collettiva sulla spesa corrente ASL (2024)')
ax.set_xlabel('Percentuale sul totale')
ax.set_ylabel('Numero di ASL')
ax.legend(frameon=False, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

figures_dir = Path.cwd() / '..' / '..' / 'preview'
figures_dir.mkdir(parents=True, exist_ok=True)
out = figures_dir / 'bdap_lea_prevenzione_pct_distribuzione.png'
plt.savefig(out, dpi=150)
plt.show()


### 5. Top 10 e Bottom 10

Le ASL con la quota piu' alta e piu' bassa di prevenzione.

In [6]:
top10 = df.nlargest(10, 'pct_prev')[['descrizione_ente', 'pct_prev']]
bottom10 = df.nsmallest(10, 'pct_prev')[['descrizione_ente', 'pct_prev']]

print('Top 10 ASL per % prevenzione')
display(top10)
print()
print('Bottom 10 ASL per % prevenzione')
display(bottom10)


### 6. Caveat

- **Perimetro**: include solo ASL/USL/ASP (70 enti). Escluse AO, AOU, enti regionali/atipici.
- **Consuntivo 2024**: ultimo anno completo, ma alcuni enti potrebbero avere dati non ancora consolidati.
- **Voce contabile**: la voce "Prevenzione Collettiva e Sanita' Pubblica" e' un aggregato contabile. Non include attivita' di prevenzione svolte in altri capitoli (es. screening integrati nell'assistenza distrettuale).
- **Soglia 5%**: e' un riferimento minimo previsto dai LEA, non un obiettivo vincolante o una misura di efficacia.
- **Nessuna correzione pro-capite**: qui si guarda la quota percentuale, non la spesa per abitante. Un'ASL piccola puo' avere una quota alta ma un volume assoluto basso.